# 08 - A_v (vertex/star) symmetry breaking vs hz

How well do the trained NQS obey the exact vertex symmetry `psi(A_v . sigma) == psi(sigma)`?
At `hz=0` it is exact; at `hz != 0` the field does not commute with `A_v`, so it is
*physically* broken and the non-symmetric Block 1 reproduces that breaking.

Data: `results/symmetry/sym_L*_hz*.json` produced by `scripts/symmetry_check.py`.
Per run: metric summaries + `delta` histograms, for Born (sigma~|psi|^2) and uniform
configs, split into bulk (clean 4-edge generators) and edge vertex stars.

Key quantities (see `symmetry_breaking_metric` in the script):
`delta_v = log psi(A_v s) - log psi(s)`, `r_v = exp(delta_v)`; and the Born anchor
`<A_v> = E_born[r_v]` (=1 at hz=0, dropping with hz).

In [ ]:
import json, glob, os
import numpy as np
import matplotlib.pyplot as plt

SYM_DIR = "../results/symmetry"
FIG_DIR = "../figures"
os.makedirs(FIG_DIR, exist_ok=True)

def load(arm="cnn-baseline", L=4):
    runs = []
    for f in sorted(glob.glob(os.path.join(SYM_DIR, "sym_L*_hz*.json"))):
        d = json.load(open(f))
        if d.get("arm", "cnn-baseline") != arm or d["L"] != L:
            continue
        runs.append(d)
    runs.sort(key=lambda d: d["hz"])
    return runs

def series(runs, tag, vset, key):
    """(hz, value) arrays for runs that carry tag_vset (e.g. born_bulk)."""
    xs, ys = [], []
    for r in runs:
        blk = r.get(f"{tag}_{vset}")
        if blk and key in blk["metric"]:
            xs.append(r["hz"]); ys.append(blk["metric"][key])
    return np.array(xs), np.array(ys)

L = 4
runs = load(arm="cnn-baseline", L=L)
print(f"loaded {len(runs)} runs at L={L}:", [round(r['hz'], 3) for r in runs])

## Fig A - symmetry-breaking metric vs hz (Born primary, uniform stress-test)
Expect ~0 (numerical/MCMC floor) at hz=0, rising monotonically with hz.

In [ ]:
KEY = "mean_abs_delta"  # primary metric; swap for 'rms_delta' / 'mean_abs_ratio_dev'
fig, ax = plt.subplots(figsize=(6, 4))
for tag, style in (("born", "-o"), ("unif", "--s")):
    x, y = series(runs, tag, "bulk", KEY)
    if len(x):
        ax.plot(x, y, style, label=f"{tag} (bulk)")
ax.set_xlabel("$h_z$")
ax.set_ylabel(r"$\langle|\delta_v|\rangle$  (mean abs log-amp deviation)")
ax.set_title(f"$A_v$ symmetry breaking vs $h_z$  (L={L})")
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/08_breaking_vs_hz_L{L}.png", dpi=150)
plt.show()

## Fig B - physical anchor: `<A_v>` vs hz (Born ratio mean)
`<A_v>` = mean of the Born-weighted amplitude ratio. Should be ~1 at hz=0 and drop as
the field disorders the vertex stars -- an independent cross-check of the pipeline.

In [ ]:
x, y = series(runs, "born", "bulk", "Av_mean")
_, yerr = series(runs, "born", "bulk", "Av_sem")
fig, ax = plt.subplots(figsize=(6, 4))
if len(x):
    ax.errorbar(x, y, yerr=yerr, fmt="-o", capsize=3)
ax.axhline(1.0, ls=":", c="k", alpha=0.5)
ax.set_xlabel("$h_z$"); ax.set_ylabel(r"$\langle A_v\rangle$  (Born ratio mean)")
ax.set_title(f"Vertex-stabilizer expectation vs $h_z$  (L={L})")
ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/08_Av_vs_hz_L{L}.png", dpi=150)
plt.show()

## Fig C - distribution of `delta_v` across hz (Born, bulk)
Spike at 0 for hz=0, broadening as the symmetry breaks.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
cmap = plt.cm.viridis
n = max(len(runs) - 1, 1)
for i, r in enumerate(runs):
    blk = r.get("born_bulk")
    if not blk:
        continue
    edges = np.asarray(blk["hist"]["edges"]); counts = np.asarray(blk["hist"]["counts"], float)
    if counts.sum() == 0:
        continue
    centers = 0.5 * (edges[:-1] + edges[1:]); w = np.diff(edges)
    dens = counts / counts.sum() / w
    ax.plot(centers, dens, color=cmap(i / n), label=f"$h_z$={r['hz']:.2f}")
ax.set_xlabel(r"$\delta_v = \log\psi(A_v\sigma) - \log\psi(\sigma)$")
ax.set_ylabel("density (Born, bulk)"); ax.set_yscale("log")
ax.set_title(f"$\\delta_v$ distribution vs $h_z$  (L={L})")
ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=2)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/08_delta_dist_L{L}.png", dpi=150)
plt.show()